# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/api/python/) library, based on its Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

This dataset contains ordered logistic regression outputs with log likelihoods, coefficients, standard errors, and p-values for predictors of adoption of indigenous and modern knowledge by pastoralist households in Northern Kenya.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List all available record sets in the dataset and preview their fields and IDs. For every entity, we use its `@id` as reference.

In [ ]:
# Show available record sets and their field info, referencing by `@id`.
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets:")
rs_ids = []
for rs in record_sets:
    print(f"- Record set @id: {rs['@id']}")
    rs_ids.append(rs['@id'])
    if 'field' in rs:
        print("  Fields:")
        # croissant 1.0: each field is a dict with `@id` and others
        for f in rs['field']:
            print(f"   - {f['@id']}: {f.get('description', f.get('name', ''))}")
    print()

## 3. Data Extraction
Load records of each record set into DataFrames for further analysis. Record set and field IDs are referenced by their `@id`.

In [ ]:
# Load all records from each record set into a DataFrame, using @id to reference each.
import collections

# Prepare dict of DataFrames per record set (by @id)
dataframes = collections.OrderedDict()
for record_set in rs_ids:
    try:
        records_iter = dataset.records(record_set=record_set)
        records = list(records_iter)
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded {len(df)} rows for record set '{record_set}'. Columns:", df.columns.tolist())
    except Exception as e:
        print(f"Failed to load record set '{record_set}':", e)

# Examine the columns of the first DataFrame for demonstration
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"\nColumns in record set '{first_rs_id}': {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We demonstrate numeric field filtering and normalization using column `@id`. Replace the variables below with actual record set and field `@id`s relevant to your data.

**Example:** If the column `@id` is `'http://mlcommons.org/croissant/field/log_likelihood'`, then use that exact string for field access.

In [ ]:
# Example: Filter and normalize 'log_likelihood' field, using its @id
# You may update the chosen record set/field IDs if needed (based on the overview above)

# ========== Update these variables with relevant @ids as found above ============
record_set_id = rs_ids[0] if rs_ids else None
# Select the first numeric field (replace with a real @id from your data)
if record_set_id and not dataframes[record_set_id].empty:
    numeric_cols = dataframes[record_set_id].select_dtypes(include='number').columns
    if len(numeric_cols) == 0:
        # Try to coerce all columns, in case they are strings
        df_temp = dataframes[record_set_id].copy()
        df_temp = df_temp.apply(pd.to_numeric, errors='ignore')
        numeric_cols = df_temp.select_dtypes(include='number').columns
    if len(numeric_cols) == 0:
        print('No numeric columns found in this record set.')
    else:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = dataframes[record_set_id][numeric_field].mean() if pd.notnull(dataframes[record_set_id][numeric_field].mean()) else 0
        filtered_df = dataframes[record_set_id][dataframes[record_set_id][numeric_field] > threshold]
        print(f"Filtered records with '{numeric_field}' > {threshold:.2f} (using field @id '{numeric_field}'):")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nFirst few values for '{numeric_field}' and normalized:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a categorical field (choose first non-numeric field)
        cat_fields = [col for col in filtered_df.columns if col != numeric_field and filtered_df[col].dtype == object]
        group_field = cat_fields[0] if cat_fields else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field, observed=True)[numeric_field].mean()
            print(f"\nMean '{numeric_field}' grouped by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No categorical grouping field found.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships. Here, we plot the distribution of a selected numeric field and a group comparison if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric field
if record_set_id and 'numeric_field' in locals() and numeric_field in dataframes[record_set_id].columns:
    plt.figure(figsize=(8,5))
    sns.histplot(dataframes[record_set_id][numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}' (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group if category present
    if 'group_field' in locals() and group_field is not None and group_field in dataframes[record_set_id].columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[record_set_id].dropna(subset=[group_field, numeric_field]))
        plt.title(f"'{numeric_field}' by '{group_field}' (@id)")
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to:
- Load metadata and records using Croissant schema from FAIR² dataset,
- Enumerate all available record sets and their fields by `@id`,
- Extract structured records into DataFrames for processing,
- Filter and normalize a numeric field, and group by key attributes,
- Visualize variable distributions and group comparisons.

This approach enables reproducible exploration and analysis of Croissant-standard datasets, referencing all fields and record sets by their canonical `@id` as recommended.